# 🎯 Project: Expectation Decider
**Mathematics & Advanced Statistics – Probability Analysis Report**

**Name:** YOUR NAME &nbsp;|&nbsp; **GRID:** YOUR GRID

### Project Details
| | |
|---|---|
| **Objective** | Analyse probability patterns to understand which factors affect a student's chance of passing a competitive maths exam |
| **Dataset** | `student_data.csv` – 200 students (AI-generated) |
| **Fields** | `study_hours`, `attendance`, `group_discussion`, `previous_test_score`, `final_exam_pass` |
| **Topics** | Basic probability, empirical vs theoretical, random variable, Venn diagram, contingency table, independence, Bayes' theorem |
| **Tools** | Python, Pandas, Matplotlib, SciPy |

## 0. Load the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from math import comb

df = pd.read_csv("student_data.csv")
total = len(df)

print("Total students:", total)
df.head()

## 1. Understanding the Basics

**What is Probability?**  
Probability tells us how likely an event is. It is a number between 0 (impossible) and 1 (certain).

$$P(A) = \frac{\text{Favourable outcomes}}{\text{Total outcomes}}$$

**Key Terms**

| Term | Meaning |
|---|---|
| Experiment | An action with an uncertain result (picking a student at random) |
| Outcome | One possible result |
| Sample Space (S) | All possible outcomes (all 200 students) |
| Event | A part of the sample space (e.g. students who passed) |
| Complement | Event does *not* happen: P(A′) = 1 − P(A) |
| Mutually exclusive | Two events cannot happen together |
| Independent | One event does not change the probability of the other |

**Three event examples from the dataset:**

In [ ]:
pass_count  = (df.final_exam_pass == "Pass").sum()
study_count = (df.study_hours > 10).sum()
score_count = (df.previous_test_score >= 75).sum()

print(f"1) P(Pass)             = {pass_count}/{total}  = {pass_count/total:.3f}")
print(f"2) P(Study > 10 hrs)   = {study_count}/{total} = {study_count/total:.3f}")
print(f"3) P(Prev score >= 75) = {score_count}/{total}  = {score_count/total:.3f}")

## 2. Types of Events

* **Empirical probability** – calculated from observed data: (times event happened) / (total trials)
* **Theoretical probability** – calculated from equally likely outcomes, without any data

In [ ]:
# Empirical: probability of passing (from observed results)
p_empirical = pass_count / total
print(f"Empirical  P(Pass) = {pass_count}/{total} = {p_empirical:.3f}")

# Theoretical: pick a student by ID (1 to 200, all equally likely).
# Event: ID is divisible by 5 -> 5, 10, ..., 200 = 40 favourable IDs
favourable = 200 // 5
print(f"Theoretical P(ID divisible by 5) = {favourable}/{total} = {favourable/total:.3f}")

## 3. Random Variable & Probability Distribution

**Random variable X** = number of students who pass among 3 randomly selected students (X = 0, 1, 2, 3).

Each student passes with probability *p* = P(Pass), so X follows a **Binomial(3, p)** distribution:

$$P(X=k) = \binom{3}{k}\,p^k\,(1-p)^{3-k}$$

Mean = 3p &nbsp;&nbsp; Variance = 3p(1−p)

In [ ]:
p = p_empirical
q = 1 - p

x_values = [0, 1, 2, 3]
probs = [comb(3, k) * p**k * q**(3 - k) for k in x_values]

table = pd.DataFrame({"X": x_values, "P(X)": [round(v, 4) for v in probs]})
print(table.to_string(index=False))
print("Total probability =", round(sum(probs), 4))

mean = sum(k * pr for k, pr in zip(x_values, probs))
variance = sum(k**2 * pr for k, pr in zip(x_values, probs)) - mean**2

print(f"\nMean     = {mean:.4f}   (3p  = {3*p:.4f})")
print(f"Variance = {variance:.4f}   (3pq = {3*p*q:.4f})")

plt.bar(x_values, probs, color="#c00000", edgecolor="black")
plt.xticks(x_values)
plt.xlabel("Number of students passing (X)")
plt.ylabel("P(X)")
plt.title("Probability Distribution of X")
plt.show()

## 4. Venn Diagram

* **A** = students who study more than 10 hours/week  
* **B** = students who attend more than 80% of classes  
* **A ∩ B** = students who satisfy both

In [ ]:
A = df.study_hours > 10
B = df.attendance > 80

only_A  = (A & ~B).sum()
only_B  = (B & ~A).sum()
both    = (A & B).sum()
neither = (~A & ~B).sum()

print(f"Only A = {only_A} | Only B = {only_B} | Both = {both} | Neither = {neither}")
print(f"P(A) = {A.mean():.3f}, P(B) = {B.mean():.3f}, P(A and B) = {(A & B).mean():.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.add_patch(Circle((-0.6, 0), 1.2, color="#1f77b4", alpha=0.45, label="Study > 10 hrs (A)"))
ax.add_patch(Circle((0.6, 0), 1.2, color="#ff7f0e", alpha=0.45, label="Attendance > 80% (B)"))
ax.text(-1.15, 0, f"Only A\n{only_A}", ha="center", va="center", fontsize=12)
ax.text(0, 0, f"A ∩ B\n{both}", ha="center", va="center", fontsize=12, fontweight="bold")
ax.text(1.15, 0, f"Only B\n{only_B}", ha="center", va="center", fontsize=12)
ax.text(0, -1.55, f"Neither: {neither}", ha="center", fontsize=11)
ax.set_xlim(-2.2, 2.2)
ax.set_ylim(-1.8, 1.6)
ax.set_aspect("equal")
ax.axis("off")
ax.legend(loc="upper center", ncol=2)
ax.set_title("Venn Diagram: Study Hours vs Attendance")
plt.show()

## 5. Contingency Table & Probability Calculations

In [ ]:
table = pd.crosstab(df.group_discussion, df.final_exam_pass, margins=True, margins_name="Total")
table = table[["Pass", "Fail", "Total"]]
print(table)

yes_pass  = table.loc["Yes", "Pass"]
yes_total = table.loc["Yes", "Total"]
all_pass  = table.loc["Total", "Pass"]
grand     = table.loc["Total", "Total"]

joint       = yes_pass / grand          # P(GD=Yes AND Pass)
marginal    = all_pass / grand          # P(Pass)
conditional = yes_pass / yes_total      # P(Pass | GD=Yes)

print(f"\nJoint       P(Yes and Pass) = {yes_pass}/{grand}  = {joint:.4f}")
print(f"Marginal    P(Pass)         = {all_pass}/{grand} = {marginal:.4f}")
print(f"Conditional P(Pass | Yes)   = {yes_pass}/{yes_total}  = {conditional:.4f}")

pd.crosstab(df.group_discussion, df.final_exam_pass).plot(kind="bar", color=["#c00000", "#2e7d32"], edgecolor="black")
plt.title("Group Discussion vs Final Result")
plt.ylabel("Number of students")
plt.xticks(rotation=0)
plt.show()

## 6. Understanding Relationships

**Intuition:** P(Pass | Yes) answers *"If I already know a student takes part in group discussions, what is the chance they pass?"*  
We look only at the participating students (instead of all 200) and count how many of them passed. If this is higher than P(Pass), participation slightly improves the chance of passing.

**Rules to check:**
* Independent → P(A ∩ B) = P(A) × P(B)
* Mutually exclusive → P(A ∩ B) = 0

In [ ]:
p_yes = yes_total / grand

print(f"P(Yes) x P(Pass)    = {p_yes:.3f} x {marginal:.3f} = {p_yes * marginal:.4f}")
print(f"P(Yes and Pass)     = {joint:.4f}")
print(f"P(Pass | Yes)       = {conditional:.4f}   vs   P(Pass) = {marginal:.4f}")

# Chi-square test (p-value > 0.05 means the events are independent)
from scipy.stats import chi2_contingency
chi2, p_value, _, _ = chi2_contingency(pd.crosstab(df.group_discussion, df.final_exam_pass))
print(f"\nChi-square p-value = {p_value:.4f}")

if p_value > 0.05:
    print("Conclusion: INDEPENDENT (difference is small and not statistically significant)")
else:
    print("Conclusion: DEPENDENT")

**Conclusion:**  
* Not **mutually exclusive** → P(Yes ∩ Pass) = 0.395 ≠ 0 (a student can participate *and* pass).  
* P(Yes) × P(Pass) = 0.375 is very close to P(Yes ∩ Pass) = 0.395, and P(Pass | Yes) = 0.653 is close to P(Pass) = 0.62.  
* The chi-square p-value (≈ 0.30) is greater than 0.05, so the small gap is just sampling noise.  
* ✅ Group discussion and passing the exam are **independent events** in this sample.

## 7. Bayes' Theorem

Let **H** = high attendance (> 80%). Given: P(H | Pass) = 0.70, P(H | Fail) = 0.40, P(H) = 0.60.

$$P(Pass \mid H) = \frac{P(H \mid Pass) \times P(Pass)}{P(H)}$$

P(Pass) is not given, so find it from total probability:

$$0.60 = 0.70\,x + 0.40\,(1 - x) \;\Rightarrow\; x = \frac{0.60 - 0.40}{0.70 - 0.40} = \frac{2}{3}$$

In [ ]:
p_h_given_pass = 0.70
p_h_given_fail = 0.40
p_h = 0.60

# Step 1: P(Pass)
p_pass = (p_h - p_h_given_fail) / (p_h_given_pass - p_h_given_fail)
print(f"Step 1: P(Pass) = (0.60 - 0.40) / (0.70 - 0.40) = {p_pass:.4f}")

# Step 2: Bayes' theorem
answer = (p_h_given_pass * p_pass) / p_h
print(f"Step 2: P(Pass | H) = (0.70 x {p_pass:.4f}) / 0.60 = {answer:.4f}  ({answer*100:.2f}%)")

**Answer:** P(Pass | High attendance) = **7/9 ≈ 0.778 (77.8%)** – higher than the overall pass rate of 66.7%, so high attendance increases the chance of passing.

## 8. Which Factors Affect Passing the Most?

We compare the pass rate of students who meet a condition with those who do not.

In [ ]:
df["passed"] = (df.final_exam_pass == "Pass").astype(int)

rates = pd.DataFrame({
    "Yes / Above": [df[df.study_hours > 10].passed.mean(),
                    df[df.attendance > 80].passed.mean(),
                    df[df.previous_test_score >= 60].passed.mean(),
                    df[df.group_discussion == "Yes"].passed.mean()],
    "No / Below":  [df[df.study_hours <= 10].passed.mean(),
                    df[df.attendance <= 80].passed.mean(),
                    df[df.previous_test_score < 60].passed.mean(),
                    df[df.group_discussion == "No"].passed.mean()],
}, index=["Study > 10 hrs", "Attendance > 80%", "Previous score >= 60", "Group discussion"])

rates["Difference"] = rates["Yes / Above"] - rates["No / Below"]
print(rates.round(3))

rates[["Yes / Above", "No / Below"]].plot(kind="bar", color=["#2e7d32", "#c00000"], edgecolor="black", figsize=(8, 4))
plt.ylabel("P(Pass)")
plt.title("Pass Probability by Factor")
plt.xticks(rotation=15)
plt.show()

print("\nCorrelation with passing:")
print(df[["study_hours", "attendance", "previous_test_score", "passed"]].corr()["passed"].drop("passed").round(3))

## 📝 Final Summary

**Overall:** Out of 200 students, **124 passed (62%)** and 76 failed (38%).

| Factor | Pass rate (Yes) | Pass rate (No) | Difference |
|---|---|---|---|
| Study > 10 hrs/week | 69.7% | 50.6% | **+19.1%** |
| Attendance > 80% | 72.2% | 53.6% | **+18.6%** |
| Previous score ≥ 60 | 66.0% | 58.0% | +8.0% |
| Group discussion | 65.3% | 57.0% | +8.3% |

**Key findings**
1. **Study hours** is the strongest factor – students who study more than 10 hours a week pass about 19% more often (correlation 0.23).
2. **Attendance** is almost equally important – students above 80% attendance pass about 19% more often (correlation 0.21). Bayes' theorem also shows P(Pass | high attendance) ≈ 77.8% against a base rate of 66.7%.
3. **Previous test score** has a positive effect (correlation 0.21), but passing depends on more than past marks alone.
4. **Group discussion** shows only a small difference, and the chi-square test (p ≈ 0.30) says it is **independent** of passing in this sample.
5. For 3 random students, the expected number of passes is **1.86** with variance **0.707**; getting exactly 2 passes is the most likely outcome (43.8%).
6. About **28%** of students (56 of 200) both study more than 10 hours *and* attend more than 80% of classes.

**Conclusion:** The probability of passing is influenced mostly by **study hours** and **attendance**, followed by previous test score. Group discussion has little real effect. To raise pass rates, the institute should encourage regular study (10+ hours/week) and high attendance (80%+).

> Note: This is AI-generated sample data, so these conclusions apply to this dataset only.